# ml2drender quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kanahiro/ml2drender/blob/main/examples/python/quickstart.ipynb)

Render a [MapLibre style](https://maplibre.org/maplibre-style-spec/) to an image in Python: no GPU, no browser, no C libraries. Just `pip install`.

In [ ]:
%pip install -q ml2drender pyproj basemaplibre geopandas

Load a style, pick a camera, render. The renderer fetches the tiles, sprites and fonts the view needs.

In [ ]:
import httpx
import ml2drender
from IPython.display import Image

style = httpx.get("https://demotiles.maplibre.org/style.json").json()
renderer = ml2drender.Renderer(style)


def show(rgba, frame):
    return Image(ml2drender.encode_png(rgba, *frame.pixels))


view = ml2drender.View((137.0, 37.0), zoom=4, width=600, height=400)
show(await renderer.render_view(view), view)

Rotate the camera with `bearing`. Resources already loaded are reused.

In [ ]:
view = ml2drender.View((137.0, 37.0), zoom=4, width=600, height=400, bearing=30)
show(await renderer.render_view(view), view)

Draw the same style in another map projection. Pass the CRS as PROJJSON; here pyproj provides it for Equal Earth (EPSG:8857).

In [ ]:
from pyproj import CRS, Transformer

equal_earth = renderer.projected(CRS.from_epsg(8857).to_json_dict())
world = Transformer.from_crs(4326, 8857, always_xy=True).transform_bounds(-180, -90, 180, 90)
box = ml2drender.ProjectedBox(world, size=800)
show(await equal_earth.render(box), box)

Plot your own data: add a GeoJSON source and a layer to the style. Here, earthquakes of magnitude 4.5+ in the past 30 days from the [USGS feed](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php), sized and colored by magnitude with [expressions](https://maplibre.org/maplibre-style-spec/expressions/). `data` takes a GeoJSON `dict` (from a GeoDataFrame: `gdf.to_geo_dict()`) or a URL. The style is fixed for a renderer's life, so make a new one.

In [ ]:
quakes = httpx.get("https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson").json()
magnitude = ["get", "mag"]
quake_style = {
    **style,
    "sources": {**style["sources"], "quakes": {"type": "geojson", "data": quakes}},
    "layers": [
        *style["layers"],
        {
            "id": "quakes",
            "type": "circle",
            "source": "quakes",
            "paint": {
                "circle-radius": ["interpolate", ["exponential", 2], magnitude, 4.5, 2, 8, 20],
                "circle-color": ["interpolate", ["linear"], magnitude, 4.5, "#fdd49e", 6, "#ef6548", 7.5, "#7f0000"],
                "circle-opacity": 0.7,
                "circle-stroke-color": "white",
                "circle-stroke-width": 0.5,
            },
        },
    ],
}

view = ml2drender.View((150.0, 10.0), zoom=1, width=800, height=500)
show(await ml2drender.Renderer(quake_style).render_view(view), view)


Plot with GeoPandas and matplotlib: [basemaplibre](https://github.com/RaczeQ/basemaplibre) draws a MapLibre style under the axes with ml2drender, with the parameters of contextily's `add_basemap`. The style is rendered directly in the CRS of the data, so labels stay upright and sharp; here the earthquakes again, in Equal Earth.

In [ ]:
import basemaplibre as bml
import geopandas

gdf = geopandas.GeoDataFrame.from_features(quakes, crs=4326).to_crs(8857)
ax = gdf.plot(column="mag", cmap="OrRd", markersize=(gdf["mag"] - 4) ** 3 * 4, alpha=0.7, legend=True, figsize=(10, 6))
bml.add_basemap(ax, crs=gdf.crs, source="OpenFreeMap.Positron")
ax.set_axis_off()

## Next

- Save an image: `open("map.png", "wb").write(ml2drender.encode_png(rgba, width, height))`; `encode_webp` and `encode_jpeg` too.
- Authentication, caching or local files: pass an async `load(url)` to `Renderer`.
- Tiles of a CRS grid and rotated cameras in any CRS: see the [Python README](https://github.com/Kanahiro/ml2drender/tree/main/crates/python#readme).